In [2]:
#Cell 1 — Imports
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [3]:
#Cell 2 — Config / paths
# Paths
DATASET_PATH  = "/mnt/hdd/colorizer_gan/datasets/coco"
TRAIN_PATH    = os.path.join(DATASET_PATH, "color/train2017")
VAL_PATH      = os.path.join(DATASET_PATH, "val/val2017")
OUTPUT_PATH   = os.path.expanduser("~/colorizer_gan/outputs")

# Training config
IMAGE_SIZE    = 256
BATCH_SIZE    = 64       # up from 32 — 32GB VRAM can handle this easily
NUM_WORKERS   = 16       # up from 8 — 250GB RAM means no bottleneck
PIN_MEMORY    = True
PREFETCH      = 4        # prefetch 4 batches ahead, keeps GPU fed

# Make sure output folder exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Train path: {TRAIN_PATH}")
print(f"Val path:   {VAL_PATH}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

Train path: /mnt/hdd/colorizer_gan/datasets/coco/color/train2017
Val path:   /mnt/hdd/colorizer_gan/datasets/coco/val/val2017
Image size: 256x256
Batch size: 64


In [4]:
#Cell 3 — Verify dataset (count images, check integrity)
def verify_dataset(path, name="Dataset"):
    if not os.path.exists(path):
        print(f" {name} path not found: {path}")
        return 0

    files = [
        f for f in os.listdir(path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f" {name} found at: {path}")
    print(f"   Total images: {len(files)}")

    corrupt = 0
    for f in files[:50]:
        try:
            img = Image.open(os.path.join(path, f))
            img.verify()
        except Exception:
            corrupt += 1

    print(f"   Corrupt images in first 50 checked: {corrupt}")
    return len(files)

train_count = verify_dataset(TRAIN_PATH, "Train")
val_count   = verify_dataset(VAL_PATH,   "Validation")

 Train found at: /mnt/hdd/colorizer_gan/datasets/coco/color/train2017
   Total images: 118287
   Corrupt images in first 50 checked: 0
 Validation found at: /mnt/hdd/colorizer_gan/datasets/coco/val/val2017
   Total images: 5000
   Corrupt images in first 50 checked: 0


In [5]:
#Cell 4 — LAB conversion explanation + example
def show_lab_example(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    img_np = np.array(img) / 255.0

    lab = rgb2lab(img_np).astype("float32")
    L   = lab[:, :, 0]
    A   = lab[:, :, 1]
    B   = lab[:, :, 2]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_np)
    axes[0].set_title("Original RGB")
    axes[1].imshow(L, cmap="gray")
    axes[1].set_title("L channel (input)")
    axes[2].imshow(A, cmap="RdYlGn_r")
    axes[2].set_title("A channel (target)")
    axes[3].imshow(B, cmap="RdYlBu_r")
    axes[3].set_title("B channel (target)")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, "lab_example.png"), dpi=150)
    plt.close()
    print(f" LAB example saved to outputs/lab_example.png")
    print(f"L range: {L.min():.1f} to {L.max():.1f}")
    print(f"A range: {A.min():.1f} to {A.max():.1f}")
    print(f"B range: {B.min():.1f} to {B.max():.1f}")

first_image = os.path.join(TRAIN_PATH, os.listdir(TRAIN_PATH)[0])
show_lab_example(first_image)

 LAB example saved to outputs/lab_example.png
L range: 4.3 to 100.0
A range: -44.2 to 34.7
B range: -43.5 to 75.7


In [6]:
#Cell 5 — Dataset class definition
class ColorizationDataset(Dataset):
    def __init__(self, path, image_size=256, split="train"):
        self.path       = path
        self.image_size = image_size
        self.split      = split

        self.files = sorted([
            os.path.join(path, f)
            for f in os.listdir(path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        if len(self.files) == 0:
            raise RuntimeError(f"No images found in {path}")

        print(f" {split} dataset loaded: {len(self.files)} images")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self.files))

        img = img.resize((self.image_size, self.image_size), Image.BICUBIC)
        img_np = np.array(img) / 255.0

        lab = rgb2lab(img_np).astype("float32")

        L  = (lab[:, :, 0] / 50.0) - 1.0
        AB = lab[:, :, 1:] / 128.0

        L  = torch.tensor(L).unsqueeze(0)
        AB = torch.tensor(AB).permute(2, 0, 1)

        return L, AB

    def __repr__(self):
        return f"ColorizationDataset(split={self.split}, size={len(self.files)})"

In [7]:
#Cell 6 — DataLoader setup
train_dataset = ColorizationDataset(TRAIN_PATH, IMAGE_SIZE, split="train")
val_dataset   = ColorizationDataset(VAL_PATH,   IMAGE_SIZE, split="val")

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = PIN_MEMORY,
    drop_last   = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = PIN_MEMORY,
    drop_last   = False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

L_batch, AB_batch = next(iter(train_loader))
print(f"\nL batch shape:  {L_batch.shape}")
print(f"AB batch shape: {AB_batch.shape}")
print(f"L range:  {L_batch.min():.2f} to {L_batch.max():.2f}")
print(f"AB range: {AB_batch.min():.2f} to {AB_batch.max():.2f}")

 train dataset loaded: 118287 images
 val dataset loaded: 5000 images
Train batches: 1848
Val batches:   79

L batch shape:  torch.Size([64, 1, 256, 256])
AB batch shape: torch.Size([64, 2, 256, 256])
L range:  -1.00 to 1.00
AB range: -0.74 to 0.73


Below are to generate images/graphs/tabnles for the report(as my first one was't detailed enough....)

In [8]:
#Cell 7 — Visualize a sample batch (save to outputs/)
def visualize_batch(L_batch, AB_batch, n=4):
    fig, axes = plt.subplots(n, 2, figsize=(8, n * 4))
    fig.suptitle("Sample Batch — Grayscale Input vs Color Target", fontsize=14)

    for i in range(n):
        L  = (L_batch[i].squeeze().numpy() + 1.0) * 50.0
        AB = AB_batch[i].permute(1, 2, 0).numpy() * 128.0

        lab = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32")
        lab[:, :, 0]  = L
        lab[:, :, 1:] = AB

        rgb = lab2rgb(lab)
        rgb = np.clip(rgb, 0, 1)

        axes[i, 0].imshow(L, cmap="gray")
        axes[i, 0].set_title("Grayscale Input (L)")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(rgb)
        axes[i, 1].set_title("Color Target (LAB → RGB)")
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, "sample_batch.png"), dpi=150)
    plt.close()
    print(" Sample batch saved to outputs/sample_batch.png")

visualize_batch(L_batch, AB_batch, n=4)

 Sample batch saved to outputs/sample_batch.png


In [11]:
#dataset distribution graph
import random

def plot_dataset_statistics(train_path, sample_size=1000):
    files = os.listdir(train_path)
    sample = random.sample(files, min(sample_size, len(files)))
    
    widths, heights, aspects = [], [], []
    lab_l, lab_a, lab_b = [], [], []
    
    print(f"Analyzing {len(sample)} sample images...")
    
    for f in sample:
        try:
            img = Image.open(os.path.join(train_path, f)).convert("RGB")
            w, h = img.size
            widths.append(w)
            heights.append(h)
            aspects.append(w / h)
            
            # LAB stats on small resized version (faster)
            img_small = img.resize((64, 64))
            img_np    = np.array(img_small) / 255.0
            lab       = rgb2lab(img_np).astype("float32")
            lab_l.append(lab[:, :, 0].mean())
            lab_a.append(lab[:, :, 1].mean())
            lab_b.append(lab[:, :, 2].mean())
        except Exception:
            continue

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Dataset Statistics — COCO 2017 Train Sample", fontsize=16, fontweight="bold")

    # --- Row 1: Image dimension stats ---

    # Width distribution
    axes[0, 0].hist(widths, bins=30, color="#4C72B0", edgecolor="white", linewidth=0.5)
    axes[0, 0].axvline(np.mean(widths), color="red", linestyle="--", label=f"Mean: {np.mean(widths):.0f}px")
    axes[0, 0].axvline(256, color="green", linestyle="--", label="Target: 256px")
    axes[0, 0].set_title("Image Width Distribution")
    axes[0, 0].set_xlabel("Width (pixels)")
    axes[0, 0].set_ylabel("Count")
    axes[0, 0].legend()

    # Height distribution
    axes[0, 1].hist(heights, bins=30, color="#DD8452", edgecolor="white", linewidth=0.5)
    axes[0, 1].axvline(np.mean(heights), color="red", linestyle="--", label=f"Mean: {np.mean(heights):.0f}px")
    axes[0, 1].axvline(256, color="green", linestyle="--", label="Target: 256px")
    axes[0, 1].set_title("Image Height Distribution")
    axes[0, 1].set_xlabel("Height (pixels)")
    axes[0, 1].set_ylabel("Count")
    axes[0, 1].legend()

    # Aspect ratio distribution
    axes[0, 2].hist(aspects, bins=30, color="#55A868", edgecolor="white", linewidth=0.5)
    axes[0, 2].axvline(1.0, color="red", linestyle="--", label="Square (1:1)")
    axes[0, 2].axvline(np.mean(aspects), color="orange", linestyle="--", label=f"Mean: {np.mean(aspects):.2f}")
    axes[0, 2].set_title("Aspect Ratio Distribution")
    axes[0, 2].set_xlabel("Width / Height")
    axes[0, 2].set_ylabel("Count")
    axes[0, 2].legend()

    # --- Row 2: LAB channel stats ---

    # L channel
    axes[1, 0].hist(lab_l, bins=30, color="#8172B2", edgecolor="white", linewidth=0.5)
    axes[1, 0].axvline(np.mean(lab_l), color="red", linestyle="--", label=f"Mean: {np.mean(lab_l):.1f}")
    axes[1, 0].set_title("L Channel Distribution (Lightness)")
    axes[1, 0].set_xlabel("Mean L value per image")
    axes[1, 0].set_ylabel("Count")
    axes[1, 0].legend()

    # A channel
    axes[1, 1].hist(lab_a, bins=30, color="#C44E52", edgecolor="white", linewidth=0.5)
    axes[1, 1].axvline(np.mean(lab_a), color="red", linestyle="--", label=f"Mean: {np.mean(lab_a):.1f}")
    axes[1, 1].set_title("A Channel Distribution (Green-Red)")
    axes[1, 1].set_xlabel("Mean A value per image")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].legend()

    # B channel
    axes[1, 2].hist(lab_b, bins=30, color="#CCB974", edgecolor="white", linewidth=0.5)
    axes[1, 2].axvline(np.mean(lab_b), color="red", linestyle="--", label=f"Mean: {np.mean(lab_b):.1f}")
    axes[1, 2].set_title("B Channel Distribution (Blue-Yellow)")
    axes[1, 2].set_xlabel("Mean B value per image")
    axes[1, 2].set_ylabel("Count")
    axes[1, 2].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, "dataset_statistics.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print("Dataset statistics saved to outputs/dataset_statistics.png")

    # Print summary stats
    print(f"\n{'='*50}")
    print(f"  DATASET SUMMARY — {len(sample)} image sample")
    print(f"{'='*50}")
    print(f"  Width:        min={min(widths)}px  max={max(widths)}px  mean={np.mean(widths):.0f}px")
    print(f"  Height:       min={min(heights)}px  max={max(heights)}px  mean={np.mean(heights):.0f}px")
    print(f"  Aspect ratio: min={min(aspects):.2f}  max={max(aspects):.2f}  mean={np.mean(aspects):.2f}")
    print(f"  L channel:    mean={np.mean(lab_l):.2f}  std={np.std(lab_l):.2f}")
    print(f"  A channel:    mean={np.mean(lab_a):.2f}  std={np.std(lab_a):.2f}")
    print(f"  B channel:    mean={np.mean(lab_b):.2f}  std={np.std(lab_b):.2f}")
    print(f"{'='*50}")

plot_dataset_statistics(TRAIN_PATH, sample_size=1000)

Analyzing 1000 sample images...
Dataset statistics saved to outputs/dataset_statistics.png

  DATASET SUMMARY — 1000 image sample
  Width:        min=200px  max=640px  mean=580px
  Height:       min=150px  max=640px  mean=482px
  Aspect ratio: min=0.54  max=3.20  mean=1.26
  L channel:    mean=47.50  std=11.73
  A channel:    mean=1.76  std=6.12
  B channel:    mean=6.94  std=10.97


In [10]:
#dataset summary table
def print_dataset_table():
    train_files = [f for f in os.listdir(TRAIN_PATH) 
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    val_files   = [f for f in os.listdir(VAL_PATH)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    total       = len(train_files) + len(val_files)

    # Estimate dataset size on disk
    train_size  = sum(os.path.getsize(os.path.join(TRAIN_PATH, f)) 
                      for f in train_files[:100]) / 100
    train_total = (train_size * len(train_files)) / (1024**3)
    val_size    = sum(os.path.getsize(os.path.join(VAL_PATH, f))
                      for f in val_files[:100]) / 100
    val_total   = (val_size * len(val_files)) / (1024**3)

    print(f"\n{'='*65}")
    print(f"  {'DATASET SUMMARY TABLE':^61}")
    print(f"{'='*65}")
    print(f"  {'Split':<15} {'Images':>10} {'Resolution':>12} {'Est. Size':>12} {'Format':>8}")
    print(f"  {'-'*61}")
    print(f"  {'Train':<15} {len(train_files):>10,} {'256×256':>12} {train_total:>10.1f}GB {'JPEG':>8}")
    print(f"  {'Validation':<15} {len(val_files):>10,} {'256×256':>12} {val_total:>10.1f}GB {'JPEG':>8}")
    print(f"  {'-'*61}")
    print(f"  {'Total':<15} {total:>10,} {'256×256':>12} {train_total+val_total:>10.1f}GB {'JPEG':>8}")
    print(f"{'='*65}")
    print(f"\n  Training Configuration:")
    print(f"  {'Batch size':<20} {BATCH_SIZE}")
    print(f"  {'Batches per epoch':<20} {len(train_files) // BATCH_SIZE}")
    print(f"  {'Image size':<20} {IMAGE_SIZE}×{IMAGE_SIZE}")
    print(f"  {'Workers':<20} {NUM_WORKERS}")
    print(f"  {'Color space':<20} LAB")
    print(f"  {'Input channels':<20} 1 (L channel)")
    print(f"  {'Output channels':<20} 2 (AB channels)")
    print(f"{'='*65}")

print_dataset_table()


                      DATASET SUMMARY TABLE                    
  Split               Images   Resolution    Est. Size   Format
  -------------------------------------------------------------
  Train              118,287      256×256       18.3GB     JPEG
  Validation           5,000      256×256        0.7GB     JPEG
  -------------------------------------------------------------
  Total              123,287      256×256       19.1GB     JPEG

  Training Configuration:
  Batch size           64
  Batches per epoch    1848
  Image size           256×256
  Workers              16
  Color space          LAB
  Input channels       1 (L channel)
  Output channels      2 (AB channels)
